# Klasyfikacja cukrzycy: drzewo decyzyjne z imputacją zer medianą

W zbiorze Pima Indians Diabetes wartości `0` w części kolumn medycznych oznaczają brak pomiaru, a nie prawidłową wartość fizjologiczną.

Procedura:
1. Zamiana zer na `NaN` w kolumnach: `glucose`, `bp`, `skin`, `insulin`, `bmi`.
2. Podział danych na zbiór treningowy i testowy.
3. Obliczenie median wyłącznie na zbiorze treningowym.
4. Uzupełnienie braków medianą i trening drzewa decyzyjnego w jednym pipeline.

Kolumna `pregnant` nie jest imputowana, ponieważ liczba ciąż równa `0` jest wartością prawidłową.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn import metrics


In [ ]:
col_names = [
    'pregnant',
    'glucose',
    'bp',
    'skin',
    'insulin',
    'bmi',
    'pedigree',
    'age',
    'label'
]

pima = pd.read_csv(
    'diabetes.csv',
    header=None,
    names=col_names
)

pima.head()


## Zamiana nieprawidłowych zer na braki danych

Nie zamieniamy zer w każdej kolumnie. Przykładowo `pregnant = 0` jest poprawną informacją.


In [ ]:
zero_as_missing_cols = ['glucose', 'bp', 'skin', 'insulin', 'bmi']

print("Liczba zer przed zamianą:")
display((pima[zero_as_missing_cols] == 0).sum().to_frame("liczba_zer"))

pima[zero_as_missing_cols] = pima[zero_as_missing_cols].replace(0, np.nan)

print("\nLiczba braków po zamianie zer na NaN:")
display(pima[zero_as_missing_cols].isna().sum().to_frame("liczba_brakow"))


In [ ]:
feature_cols = [
    'pregnant',
    'insulin',
    'bmi',
    'age',
    'glucose',
    'bp',
    'pedigree'
]

X = pima[feature_cols]
y = pima['label']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y
)

print("Rozmiar zbioru treningowego:", X_train.shape)
print("Rozmiar zbioru testowego:", X_test.shape)


## Pipeline: mediana + drzewo decyzyjne

`SimpleImputer` uczy się median tylko z `X_train`. Dzięki temu informacje ze zbioru testowego nie przenikają do treningu modelu.


In [ ]:
model = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='median')
        ),
        (
            'classifier',
            DecisionTreeClassifier(
                max_depth=4,
                splitter='best',
                criterion='gini',
                random_state=1
            )
        )
    ]
)

model.fit(X_train, y_train)


In [ ]:
# Mediany wyznaczone wyłącznie na zbiorze treningowym
medians = pd.Series(
    model.named_steps['imputer'].statistics_,
    index=feature_cols,
    name='mediana_treningowa'
)

medians.to_frame()


In [ ]:
y_pred = model.predict(X_test)


In [ ]:
print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {metrics.precision_score(y_test, y_pred):.4f}")
print(f"Recall: {metrics.recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {metrics.f1_score(y_test, y_pred):.4f}")

print("\nMacierz pomyłek:")
print(metrics.confusion_matrix(y_test, y_pred))

print("\nRaport klasyfikacji:")
print(metrics.classification_report(y_test, y_pred))


## Kontrola: czy po imputacji pozostały braki?

Poniższy kod przekształca dane za pomocą dopasowanego imputera i sprawdza liczbę brakujących wartości.


In [ ]:
X_train_imputed = model.named_steps['imputer'].transform(X_train)
X_test_imputed = model.named_steps['imputer'].transform(X_test)

print("Braki po imputacji w zbiorze treningowym:", np.isnan(X_train_imputed).sum())
print("Braki po imputacji w zbiorze testowym:", np.isnan(X_test_imputed).sum())
